In [0]:
# Import the required Delta Lake, Python and PySpark components

from datetime import datetime

from delta.tables import DeltaTable

from pyspark.sql.functions import (
    col,
    concat_ws,
    count,
    countDistinct,
    current_timestamp,
    lit,
    sha2,
    when
)

from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType
)

In [0]:
# Receive the batch and run identifiers from the Lakeflow Job

dbutils.widgets.text(
    "batch_id",
    "2009-12",
    "Batch ID"
)

dbutils.widgets.text(
    "run_id",
    "manual-run-001",
    "Run ID"
)

batch_id = dbutils.widgets.get("batch_id")
run_id = dbutils.widgets.get("run_id")


# Define the source path and target tables

source_file_path = (
    f"/Volumes/online_retail/bronze/source_files/"
    f"online_retail_{batch_id}.csv"
)

target_table = "online_retail.bronze.transactions_raw"
control_table = "online_retail.control.pipeline_runs"
layer_name = "bronze"

print(f"Run ID: {run_id}")
print(f"Batch ID: {batch_id}")
print(f"Source file: {source_file_path}")
print(f"Target table: {target_table}")

Run ID: bronze-cleanup-test-001
Batch ID: 2010-02
Source file: /Volumes/online_retail/bronze/source_files/online_retail_2010-02.csv
Target table: online_retail.bronze.transactions_raw


In [0]:
# Validate that batch_id is a valid month in exact YYYY-MM format

try:
    batch_month = datetime.strptime(batch_id, "%Y-%m")

    if batch_month.strftime("%Y-%m") != batch_id:
        raise ValueError

except ValueError as error:
    raise ValueError(
        f"Invalid batch_id: {batch_id}. Expected YYYY-MM."
    ) from error

else:
    print(f"Valid batch ID: {batch_id}")

Valid batch ID: 2010-02


In [0]:
# Define an explicit schema using the original CSV column names

source_schema = StructType([
    StructField("Invoice", StringType(), True),
    StructField("StockCode", StringType(), True),
    StructField("Description", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("InvoiceDate", TimestampType(), True),
    StructField("Price", DoubleType(), True),
    StructField("Customer ID", StringType(), True),
    StructField("Country", StringType(), True),
    StructField("source_sheet", StringType(), True),
    StructField("source_row_number", LongType(), True),
    StructField("batch_id", StringType(), True)
])

In [0]:
# Record that Bronze processing has started

if not spark.catalog.tableExists(control_table):
    raise ValueError(
        f"Control table does not exist: {control_table}"
    )


started_audit_df = (
    spark.range(1)
    .select(
        lit(run_id).alias("run_id"),
        lit(batch_id).alias("batch_id"),
        lit(layer_name).alias("layer_name"),
        lit("STARTED").alias("status"),
        current_timestamp().alias("start_timestamp"),
        lit(None).cast("timestamp").alias("end_timestamp"),
        lit(None).cast("long").alias("input_row_count"),
        lit(None).cast("long").alias("output_row_count"),
        lit(None).cast("string").alias("error_message")
    )
)


control_delta_table = DeltaTable.forName(
    spark,
    control_table
)


(
    control_delta_table.alias("target")
    .merge(
        started_audit_df.alias("source"),
        """
        target.run_id = source.run_id
        AND target.batch_id = source.batch_id
        AND target.layer_name = source.layer_name
        """
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print(
    f"Bronze processing started for batch {batch_id}, "
    f"run {run_id}."
)

Bronze processing started for batch 2010-02, run bronze-cleanup-test-001.


In [0]:
# Read the selected monthly CSV using the explicit schema

source_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .schema(source_schema)
    .load(source_file_path)
)

In [0]:
# Standardize column names and add ingestion metadata

bronze_df = (
    source_df
    .select(
        col("Invoice").alias("invoice"),
        col("StockCode").alias("stock_code"),
        col("Description").alias("description"),
        col("Quantity").alias("quantity"),
        col("InvoiceDate").alias("invoice_date"),
        col("Price").alias("price"),
        col("Customer ID").alias("customer_id"),
        col("Country").alias("country"),
        col("source_sheet"),
        col("source_row_number"),
        col("batch_id"),
        col("_metadata.file_name").alias("source_file_name"),
        col("_metadata.file_path").alias("source_file_path")
    )
    .withColumn(
        "ingestion_timestamp",
        current_timestamp()
    )
    .withColumn(
        "record_id",
        sha2(
            concat_ws(
                "||",
                col("source_sheet"),
                col("source_row_number").cast("string")
            ),
            256
        )
    )
)

In [0]:
# Validate row counts and record IDs before writing

bronze_profile = (
    bronze_df
    .agg(
        count("*").alias("bronze_row_count"),
        countDistinct("record_id").alias(
            "distinct_record_id_count"
        ),
        count(
            when(
                col("record_id").isNull(),
                1
            )
        ).alias("null_record_id_count")
    )
    .first()
)

bronze_row_count = bronze_profile["bronze_row_count"]

distinct_record_id_count = (
    bronze_profile["distinct_record_id_count"]
)

null_record_id_count = (
    bronze_profile["null_record_id_count"]
)

duplicate_record_id_count = (
    bronze_row_count - distinct_record_id_count
)


print(f"Bronze batch rows: {bronze_row_count:,}")
print(f"Distinct record IDs: {distinct_record_id_count:,}")
print(f"Duplicate record IDs: {duplicate_record_id_count:,}")
print(f"Null record IDs: {null_record_id_count:,}")


if bronze_row_count == 0:
    raise ValueError(
        f"Source batch {batch_id} contains no records."
    )

if null_record_id_count > 0:
    raise ValueError(
        f"Found {null_record_id_count} null record IDs "
        f"in batch {batch_id}."
    )

if duplicate_record_id_count > 0:
    raise ValueError(
        f"Found {duplicate_record_id_count} duplicate "
        f"record IDs in batch {batch_id}."
    )

Bronze batch rows: 29,388
Distinct record IDs: 29,388
Duplicate record IDs: 0
Null record IDs: 0


In [0]:
# Confirm that every source row belongs to the selected batch

batch_mismatch_count = (
    bronze_df
    .filter(
        col("batch_id").isNull()
        | (col("batch_id") != batch_id)
    )
    .count()
)


if batch_mismatch_count > 0:
    raise ValueError(
        f"Found {batch_mismatch_count} rows that do not "
        f"match batch {batch_id}."
    )

print(f"All source rows match batch ID: {batch_id}")

All source rows match batch ID: 2010-02


In [0]:
# Insert only previously unseen records into the Bronze Delta table

if spark.catalog.tableExists(target_table):
    bronze_delta_table = DeltaTable.forName(
        spark,
        target_table
    )

    (
        bronze_delta_table.alias("target")
        .merge(
            bronze_df.alias("source"),
            "target.record_id = source.record_id"
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(
        f"Merged batch {batch_id} into {target_table}."
    )

else:
    (
        bronze_df
        .write
        .format("delta")
        .saveAsTable(target_table)
    )

    print(
        f"Created {target_table} with batch {batch_id}."
    )

Merged batch 2010-02 into online_retail.bronze.transactions_raw.


In [0]:
# Verify the stored batch after the Delta write

stored_bronze_batch_df = (
    spark.table(target_table)
    .filter(col("batch_id") == batch_id)
)


stored_bronze_profile = (
    stored_bronze_batch_df
    .agg(
        count("*").alias("stored_batch_row_count"),
        countDistinct("record_id").alias(
            "stored_distinct_record_id_count"
        )
    )
    .first()
)

stored_batch_row_count = (
    stored_bronze_profile["stored_batch_row_count"]
)

stored_distinct_record_id_count = (
    stored_bronze_profile[
        "stored_distinct_record_id_count"
    ]
)

stored_duplicate_record_id_count = (
    stored_batch_row_count
    - stored_distinct_record_id_count
)


source_records_missing_from_bronze = (
    bronze_df
    .select("record_id")
    .join(
        stored_bronze_batch_df.select("record_id"),
        on="record_id",
        how="left_anti"
    )
    .count()
)


print(f"Stored Bronze batch rows: {stored_batch_row_count:,}")
print(
    "Stored distinct record IDs: "
    f"{stored_distinct_record_id_count:,}"
)
print(
    "Stored duplicate record IDs: "
    f"{stored_duplicate_record_id_count:,}"
)
print(
    "Source records missing from Bronze: "
    f"{source_records_missing_from_bronze:,}"
)


if stored_batch_row_count != bronze_row_count:
    raise ValueError(
        f"Stored Bronze count for batch {batch_id} does not "
        f"match the prepared source count."
    )

if stored_duplicate_record_id_count > 0:
    raise ValueError(
        f"Duplicate record IDs exist in the stored Bronze "
        f"batch {batch_id}."
    )

if source_records_missing_from_bronze > 0:
    raise ValueError(
        f"{source_records_missing_from_bronze} source records "
        f"are missing from Bronze."
    )

print(f"Stored Bronze batch {batch_id} passed validation.")

Stored Bronze batch rows: 29,388
Stored distinct record IDs: 29,388
Stored duplicate record IDs: 0
Source records missing from Bronze: 0
Stored Bronze batch 2010-02 passed validation.


In [0]:
# Mark Bronze processing as successful in the control table

control_delta_table.update(
    condition=(
        (col("run_id") == run_id)
        & (col("batch_id") == batch_id)
        & (col("layer_name") == layer_name)
    ),
    set={
        "status": lit("SUCCESS"),
        "end_timestamp": current_timestamp(),
        "input_row_count": lit(bronze_row_count),
        "output_row_count": lit(stored_batch_row_count),
        "error_message": lit(None).cast("string")
    }
)

print(
    f"Audit completed successfully for Bronze, "
    f"batch {batch_id}, run {run_id}."
)

Audit completed successfully for Bronze, batch 2010-02, run bronze-cleanup-test-001.
